In [1]:
import pandas as pd
import numpy as np

In [2]:
movies = pd.read_csv("tmdb_5000_credits.csv")
credits = pd.read_csv("tmdb_5000_movies.csv")

In [3]:
movies = movies.merge(credits, on='title')

In [ ]:
movies.head()

In [5]:
#genres
#id
#keywords
#title
#overview
#cast
#crew

movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

In [6]:
movies.duplicated().sum()
movies.dropna(inplace=True)

In [7]:
#['Action, "Adventure", "Fantasy", "Sci Fi"]
import ast
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [8]:
movies['genres'] = movies['genres'].apply(convert)

In [9]:
movies['keywords'] = movies['keywords'].apply(convert)

In [10]:
import ast
def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter+=1
        else:
            break
    return L

In [11]:
movies['cast'] = movies['cast'].apply(convert3)

In [12]:
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

In [13]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [14]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [15]:
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])

In [16]:
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

In [17]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [18]:
new_df = movies[['movie_id', 'title', 'tags']]

In [22]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [20]:
new_df.loc[:, 'tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [21]:
new_df.loc[:, 'tags'] = new_df['tags'].apply(lambda x: x.lower())

In [23]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')

In [ ]:
vectors = cv.fit_transform(new_df['tags'])
vectors.toarray().shape
print(vectors)

In [25]:
import nltk
print(nltk.__version__)

3.10.3


In [ ]:
%pip install nltk

In [26]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [27]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

In [28]:
new_df['tags'] = new_df['tags'].apply(stem)

In [29]:
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
similarity = cosine_similarity(vectors)

In [31]:
sorted(list(enumerate(similarity[0])), reverse=True, key=lambda x: x[1])[1: 6]

[(539, np.float64(0.25038669783359574)),
 (1192, np.float64(0.24779731389167606)),
 (507, np.float64(0.24283093212859141)),
 (260, np.float64(0.2409900932515112)),
 (1214, np.float64(0.23939494881986934))]

In [32]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1: 6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)
    

In [34]:
recommend('Batman')

Batman
Batman & Robin
The Dark Knight Rises
Batman Begins
Batman Returns


In [35]:
import pickle

# Export the dataframe as a dictionary
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))

# Export the similarity matrix
pickle.dump(similarity, open('similarity.pkl', 'wb'))
